# Gold Layer - Dimensional Model

Builds `dim_driver`, `dim_vehicle`, `dim_zones`, `dim_capability`, `bridge_driver_vehicle`, and `fact_trips` from `silver_taxi`.

In [0]:
USE CATALOG students_data;
USE SCHEMA `git-happens-schema`;

In [0]:
CREATE OR REPLACE TABLE dim_driver AS
SELECT
  Driver AS driver_id,
  COUNT(DISTINCT Booking_ID) AS total_trips,
  COUNT(DISTINCT Vehicle) AS vehicles_used,
  MIN(Pickup_Due) AS first_trip,
  MAX(Pickup_Due) AS last_trip
FROM silver_taxi
WHERE Driver IS NOT NULL
GROUP BY Driver;

In [0]:
CREATE OR REPLACE TABLE dim_vehicle AS
SELECT
  Vehicle AS vehicle_id,
  REGEXP_EXTRACT(Vehicle, '^([A-Z]+)', 1) AS vehicle_zone,
  REGEXP_EXTRACT(Vehicle, '(\\d+)$', 1) AS vehicle_number,
  COUNT(DISTINCT Booking_ID) AS total_trips,
  COUNT(DISTINCT Driver) AS drivers_assigned
FROM silver_taxi
WHERE Vehicle IS NOT NULL
GROUP BY Vehicle;

In [0]:
CREATE OR REPLACE TABLE bridge_driver_vehicle AS
SELECT
  Driver AS driver_id,
  Vehicle AS vehicle_id,
  COUNT(DISTINCT Booking_ID) AS trip_count,
  MIN(Pickup_Due) AS first_together,
  MAX(Pickup_Due) AS last_together
FROM silver_taxi
WHERE Driver IS NOT NULL AND Vehicle IS NOT NULL
GROUP BY Driver, Vehicle;

In [0]:
CREATE OR REPLACE TABLE dim_capability AS
SELECT * FROM VALUES
  ('6', '6 Seater',       'Capacity'),
  ('7', '7 Seater',       'Capacity'),
  ('8', '8 Seater',       'Capacity'),
  ('M', 'Minibus',        'Capacity'),
  ('W', 'Wheelchair',     'Accessibility'),
  ('Z', 'Card Reader',    'Payment'),
  ('F', 'Female',         'Customer Preference'),
  ('P', 'Pet',            'Customer Preference'),
  ('V', 'VIP',            'Premium Service'),
  ('T', 'Tour',           'Journey Purpose'),
  ('D', 'Delivery',       'Journey Purpose'),
  ('H', 'High Car',       'Vehicle Specification'),
  ('L', 'Low Car',        'Vehicle Specification')
AS t(capability_code, capability_name, business_category);

In [0]:
CREATE OR REPLACE TABLE dim_zones AS
WITH zone_list AS (
  SELECT DISTINCT zone FROM (
    SELECT Pickup_Zone AS zone FROM silver_taxi WHERE Pickup_Zone IS NOT NULL
    UNION
    SELECT Destination_Zone AS zone FROM silver_taxi WHERE Destination_Zone IS NOT NULL
  )
),
zone_stats AS (
  SELECT
    z.zone,
    -- Structural attributes
    TRIM(REGEXP_EXTRACT(z.zone, '^([A-Za-z/]+)', 1))  AS zone_prefix,
    CAST(REGEXP_EXTRACT(z.zone, '(\\d+)$', 1) AS INT) AS zone_number,

    -- Trip volumes
    COALESCE(p.pickup_count, 0)  AS pickup_count,
    COALESCE(d.dest_count, 0)    AS destination_count,
    COALESCE(p.pickup_count, 0) + COALESCE(d.dest_count, 0) AS total_trips,

    -- Pickup share
    ROUND(
      COALESCE(p.pickup_count, 0) * 100.0 /
      NULLIF(COALESCE(p.pickup_count, 0) + COALESCE(d.dest_count, 0), 0),
    1) AS pickup_pct

  FROM zone_list z
  LEFT JOIN (
    SELECT Pickup_Zone AS zone, COUNT(*) AS pickup_count
    FROM silver_taxi WHERE Pickup_Zone IS NOT NULL GROUP BY Pickup_Zone
  ) p ON z.zone = p.zone
  LEFT JOIN (
    SELECT Destination_Zone AS zone, COUNT(*) AS dest_count
    FROM silver_taxi WHERE Destination_Zone IS NOT NULL GROUP BY Destination_Zone
  ) d ON z.zone = d.zone
)
SELECT
  zone            AS zone_id,
  zone_prefix,
  zone_number,
  pickup_count,
  destination_count,
  total_trips,
  pickup_pct,

  -- Activity tier based on total trip volume
  CASE
    WHEN total_trips >= 50000  THEN 'High'
    WHEN total_trips >= 10000  THEN 'Medium'
    ELSE 'Low'
  END AS activity_tier,

  -- Primary role of the zone
  CASE
    WHEN pickup_pct >= 70 THEN 'Pickup-Dominant'
    WHEN pickup_pct <= 30 THEN 'Destination-Dominant'
    ELSE 'Balanced'
  END AS zone_role,

  -- Zone type classification
  CASE
    WHEN zone_prefix = 'DEP' THEN 'Operations'
    WHEN zone_prefix = 'AIR' THEN 'Transport Hub'
    WHEN zone_prefix IN ('BEL', 'NI', 'NAT', 'DON', 'LIM', 'DEL') THEN 'Out-of-Area'
    ELSE 'Local'
  END AS zone_type

FROM zone_stats
ORDER BY zone_number;

In [0]:
CREATE OR REPLACE TABLE fact_trips AS
SELECT
  Booking_ID,
  Driver AS driver_id,
  Vehicle AS vehicle_id,
  Status,
  Payment_Type,
  Priority,
  Capabilities,
  Booking_source,
  TRY_CAST(REPLACE(Price, ',', '') AS DOUBLE) AS price,
  TRY_CAST(REPLACE(Distance, ',', '') AS DOUBLE) AS distance,
  Pickup_Due,
  Completed,
  Time_Dispatched,
  Time_Vehicle_Arrived,
  Time_Picked_Up,
  -- Derived time metrics (minutes)
  ROUND(TIMESTAMPDIFF(MINUTE, Time_Dispatched, Time_Vehicle_Arrived), 2) AS dispatch_to_arrival_mins,
  ROUND(TIMESTAMPDIFF(MINUTE, Time_Vehicle_Arrived, Time_Picked_Up), 2) AS wait_at_pickup_mins,
  ROUND(TIMESTAMPDIFF(MINUTE, Time_Picked_Up, Completed), 2) AS trip_duration_mins,
  Pickup_Zone,
  Destination_Zone,
  Pickup_Latitude,
  Pickup_Longitude,
  Destination_Latitude,
  Destination_Longitude,
  Booked_by,
  ingestion_timestamp
FROM silver_taxi;

In [0]:
SELECT 'dim_capability' AS table_name, COUNT(*) AS row_count FROM dim_capability
UNION ALL
SELECT 'dim_driver', COUNT(*) FROM dim_driver
UNION ALL
SELECT 'dim_vehicle', COUNT(*) FROM dim_vehicle
UNION ALL
SELECT 'dim_zones', COUNT(*) FROM dim_zones
UNION ALL
SELECT 'bridge_driver_vehicle', COUNT(*) FROM bridge_driver_vehicle
UNION ALL
SELECT 'fact_trips', COUNT(*) FROM fact_trips;

In [0]:
DROP TABLE IF EXISTS dim_drivers;
DROP TABLE IF EXISTS dim_vehicles;

In [0]:
SELECT * FROM students_data.`git-happens-schema`.dim_driver LIMIT 10;

In [0]:
SELECT * FROM students_data.`git-happens-schema`.dim_vehicle LIMIT 10;

In [0]:
SELECT * FROM students_data.`git-happens-schema`.bridge_driver_vehicle LIMIT 10;

In [0]:
SELECT * FROM students_data.`git-happens-schema`.dim_capability LIMIT 10;

In [0]:
SELECT * FROM students_data.`git-happens-schema`.fact_trips LIMIT 10;